# AI Video Dubber -- try it on your own video

Upload a short English video clip and this notebook will dub it into Hindi --
**cloning each speaker's own voice**, fitting each line's timing to the
original cadence, with subtitles burned in. Runs entirely on this Colab
session's free GPU, nothing leaves Google's infrastructure.

Repo: https://github.com/CHANDRESH2002/ai-video-dubber

**Before you start**: `Runtime -> Change runtime type -> T4 GPU` (should
already be set from this notebook's config, but double-check).

**First run takes ~15-20 minutes** (downloading models + two Python
environments) -- that's one-time setup for this session, not per-video.

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Clone the pipeline

In [ ]:
!git clone https://github.com/CHANDRESH2002/ai-video-dubber.git
%cd ai-video-dubber

## 3. Set up the two environments

Two separate venvs, not one -- the voice-cloning TTS model (Chatterbox)
pins a different `transformers`/`torch` than the diarization+translation
stack, and the two genuinely conflict. See `CLAUDE.md` in the repo for why.
This mirrors exactly how the pipeline runs on a real GPU box; Colab is just
another Linux machine to it.

In [ ]:
# Main environment: diarization, transcription, translation, assembly
# --without-pip + manual get-pip.py: Colab's ensurepip is broken/incomplete
# on some images, which makes a plain `python3 -m venv` silently fail to
# install pip into the new venv -- this routes around that entirely.
!python3 -m venv .venv_main --without-pip
!curl -sS https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!./.venv_main/bin/python3 get-pip.py -q
!./.venv_main/bin/pip install -q --extra-index-url https://download.pytorch.org/whl/cu126 -r requirements.txt

In [ ]:
# Chatterbox environment: voice-cloned TTS synthesis, isolated on purpose
!python3 -m venv .venv_chatterbox --without-pip
!./.venv_chatterbox/bin/python3 get-pip.py -q
!./.venv_chatterbox/bin/pip install -q chatterbox-tts
# chatterbox-tts's watermarking dependency (resemble-perth) imports
# pkg_resources, which newer setuptools removed -- pin around it.
!./.venv_chatterbox/bin/pip install -q "setuptools<81"

## 4. Hugging Face token

The diarization model (pyannote) is gated -- you need your **own** free HF
token with access accepted on these two model pages (one-time, ~1 minute):

- https://huggingface.co/pyannote/speaker-diarization-3.1
- https://huggingface.co/pyannote/wespeaker-voxceleb-resnet34-LM

Then get a token (read access is enough) from
https://huggingface.co/settings/tokens and paste it below -- it's entered
privately (not shown/stored in the notebook file) and only used locally in
this Colab session.

In [ ]:
from getpass import getpass
hf_token = getpass("Paste your Hugging Face token and press Enter: ")
with open(".env", "w") as f:
    f.write(f"HF_TOKEN={hf_token}\n")
print("Saved.")

## 5. Upload your video

In [ ]:
from google.colab import files
import os

os.makedirs("input", exist_ok=True)
uploaded = files.upload()
video_filename = next(iter(uploaded))
video_path = os.path.join("input", video_filename)
os.rename(video_filename, video_path)
print(f"Uploaded: {video_path}")

## 6. Run the dub

`target_lang` defaults to `hi` (Hindi). `num_speakers` can be left blank to
auto-detect, or set to a specific count if you know it (more reliable for
short/noisy clips).

In [ ]:
target_lang = "hi"  # @param {type:"string"}
num_speakers = ""  # @param {type:"string"}

output_path = "output/dubbed.mp4"
os.makedirs("output", exist_ok=True)

cmd = f"./dub_from_video.sh {video_path} {output_path} {target_lang}"
if num_speakers.strip():
    cmd += f" {num_speakers.strip()}"

!{cmd}

## 7. Watch and download the result

In [ ]:
from IPython.display import Video, display
display(Video(output_path, embed=True, width=640))

In [ ]:
from google.colab import files
files.download(output_path)

---
This is a solo/unfunded research project, still early -- expect rough
edges on some clips (see `CLAUDE.md`'s Known Issues in the repo). Issues
and contributions welcome: https://github.com/CHANDRESH2002/ai-video-dubber